# Section 5: Ollama's Decision Endpoint and Nimble

**What you'll learn**
- Run decision models in Ollama next to chat models
- Call `/v1/systemone` three ways: raw HTTP, `ollama.systemone`, the TypeSafe SDK
- Read Noul, Choice and Score answers, and what Ollama's `confidence` measures
- Use case: check outbound messages for personal data, then send or hold for review
- Hit the limits on purpose: 64 questions, the prompt fit, 64 KB, text only
- Compare nimble, tev1:0.8b and hosted Jev on the same question

## Setup
- Slides 1-3
- Ollama 0.40.4 or newer, server on port 11434

In [1]:
!ollama --version

ollama version is 0.40.5


In [2]:
!ollama list

NAME                             ID              SIZE      MODIFIED       
tev1:0.8b                        abc31fdb0bf5    833 MB    7 minutes ago     
nimble:latest                    6aa3fde96874    9.5 GB    7 minutes ago     
tev1:latest                      7954dce22bc6    4.6 GB    20 minutes ago    
qwen3.5:0.8b                     de63045f2975    1.3 GB    4 days ago        
gemma4:12b                       6114515d63c1    8.0 GB    4 days ago        
qwen3.5:4b                       d8b0f5e9760c    3.3 GB    5 days ago        
qwen3.8:27b                      aaee06c39dcf    17 GB     2 weeks ago       
ornith-1.5:9b                    e5df7dcdd8a2    6.6 GB    6 weeks ago       
gemma4:12b-it-q4_K_M             4eb23ef187e2    7.6 GB    7 weeks ago       
granite4.2:30b-q4_K_M            e65d8fa47f66    17 GB     7 weeks ago       
granite4.2:8b-q4_K_M             efbf7a204b67    5.3 GB    7 weeks ago       
granite4.2:3b                    7a65e6414ad6    2.2 GB    7 weeks 

In [3]:
import time
import httpx
import numpy as np
import ollama
import pandas as pd
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Choice, Noul

## Pull a Decision Model
- Slides 4-8, covering the Model Files and Their Makers too
- `ollama.pull` is `ollama pull` from Python; an already pulled model finishes in seconds

In [4]:
ollama.pull("nimble")

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [5]:
ollama.pull("tev1:0.8b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

- `Capabilities: decision` marks a decision model
- `num_ctx` is the prompt fit per question; `quantization` is how the weights are stored

In [6]:
!ollama show nimble

  Model
    architecture        qwen35    
    parameters          9.0B      
    context length      262144    
    embedding length    4096      
    quantization        Q8_0      
    requires            0.40.4    

  Capabilities
    decision    

  Parameters
    num_ctx    8194    

  License
    Apache License 2.0                                                 
    This model fuses two artifacts, both released under Apache-2.0:    
    ...                                                                



## First Call: Raw HTTP
- Slide 9
- `POST /v1/systemone` on port 11434, the same body shape as hosted Jev
- `state`: the text to judge; `questions`: named questions
- `timeout=60`: the first call loads the model, longer than httpx's 5 second default

In [7]:
URL = "http://localhost:11434/v1/systemone"

body = {
    "model": "nimble",
    "state": "Hi Priya, your replacement card ending 4417 ships to 22 Elm St, "
             "Springfield. Call 555-0142 if it doesn't arrive.",
    "questions": {
        "personal": {
            "type": "noul",
            "instructions": "The message contains personal data "
                            "about a specific person.",
        },
    },
}

r = httpx.post(URL, json=body, timeout=60)
r.json()

{'model': 'nimble',
 'answers': {'personal': {'type': 'noul', 'noul': 0.9707504401130009}},
 'usage': {'input_tokens': 189, 'output_tokens': 0}}

- A chat model on the same server: swap in any chat model from `ollama list`

In [8]:
httpx.post(URL, json={**body, "model": "qwen3.5:2b"}).json()

{'error': 'model "qwen3.5:2b" is not a llama.cpp decision model'}

## `ollama.systemone`: Three Question Types
- Slides 10-12
- `noul`: no options, returns P(true)
- `choice`: `criteria` is a dict of option to description; `score`: a list from lowest to highest

In [9]:
QUESTIONS = {
    "personal": {
        "type": "noul",
        "instructions": "The message contains personal data about a specific person.",
    },
    "category": {
        "type": "choice",
        "instructions": "What kind of sensitive content does this outbound message "
                        "contain?",
        "criteria": {
            "clean": "nothing sensitive",
            "personal_data": "phone, address, card digits, ID numbers, personal email, "
                             "health details",
            "internal_info": "staff discount codes, internal links, unreleased plans",
        },
    },
    "risk": {
        "type": "score",
        "instructions": "How much harm could this message do if it reached the wrong "
                        "person?",
        "criteria": ["none", "some", "serious"],
    },
}

r = ollama.systemone(model="nimble", state=body["state"], questions=QUESTIONS)
r

SystemOneResponse(model='nimble', answers={'personal': SystemOneNoulAnswer(type='noul', noul=0.9883691828821889), 'category': SystemOneChoiceAnswer(type='choice', choice='personal_data', probabilities={'clean': 0.005202001925185637, 'personal_data': 0.9785945752168494, 'internal_info': 0.016203422857964937}, confidence=0.9678918628252742), 'risk': SystemOneScoreAnswer(type='score', score=1.1378327893808913, legend={'0': 'none', '1': 'some', '2': 'serious'}, probabilities={'0': 0.012645798947802038, '1': 0.8368756127235047, '2': 0.15047858832869324}, confidence=0.7553134190852571)}, usage=SystemOneUsage(input_tokens=1128, output_tokens=0))

In [10]:
r.answers["personal"].noul

0.9883691828821889

In [11]:
(
    r.answers["category"].choice,
    r.answers["category"].probabilities,
    r.answers["category"].confidence,
)

('personal_data',
 {'clean': 0.005202001925185637,
  'personal_data': 0.9785945752168494,
  'internal_info': 0.016203422857964937},
 0.9678918628252742)

In [12]:
r.answers["risk"].score, r.answers["risk"].legend, r.answers["risk"].probabilities

(1.1378327893808913,
 {'0': 'none', '1': 'some', '2': 'serious'},
 {'0': 0.012645798947802038,
  '1': 0.8368756127235047,
  '2': 0.15047858832869324})

In [13]:
r.usage

SystemOneUsage(input_tokens=1128, output_tokens=0)

## What Ollama's `confidence` Measures
- Slides 13-14
- Choice: how far the top probability sits above an even guess, `(p_max - 1/n) / (1 - 1/n)`
- Score: one minus the average distance from the top level, measured against an even spread
- Compare each with the returned `confidence`

In [14]:
p = pd.Series(r.answers["category"].probabilities)
n = len(p)
(p.max() - 1 / n) / (1 - 1 / n), r.answers["category"].confidence

(np.float64(0.9678918628252742), 0.9678918628252742)

In [15]:
p = np.array(list(r.answers["risk"].probabilities.values()))
levels = np.arange(len(p))
distance = (p * np.abs(levels - p.argmax())).sum()
even = np.abs(levels - (len(p) - 1) / 2).mean()
1 - distance / even, r.answers["risk"].confidence

(np.float64(0.7553134190852571), 0.7553134190852571)

## Running Hosted Jev Code on Ollama
- Slide 15
- Only `base_url`, `api_key` and `model` change; Ollama ignores the key, the SDK needs one
- `TypeSafeClient()` with no arguments reads `.env` and reaches hosted Jev

In [16]:
load_dotenv()

nimble = TypeSafeClient(
    base_url="http://localhost:11434",
    api_key="local",
    model="nimble",
)
tev1 = TypeSafeClient(
    base_url="http://localhost:11434",
    api_key="local",
    model="tev1:0.8b",
)
jev = TypeSafeClient()

In [17]:
PERSONAL = Noul(
    instructions="The message contains personal data about a specific person.",
)

r = nimble.system_one(body["state"], {"personal": PERSONAL})
r

SystemOneResponse(model='nimble', usage=Usage(input_tokens=189, output_tokens=0), answers={'personal': NoulAnswer(type='noul', noul=0.9707504401130009)})

In [18]:
tev1.system_one(body["state"], {"personal": PERSONAL})

SystemOneResponse(model='tev1:0.8b', usage=Usage(input_tokens=146, output_tokens=0), answers={'personal': NoulAnswer(type='noul', noul=0.8331494091696195)})

In [19]:
jev.system_one(body["state"], {"personal": PERSONAL})

SystemOneResponse(model='typesafe/jev-1.14-20261014', usage=Usage(input_tokens=315, output_tokens=20), answers={'personal': NoulAnswer(type='noul', noul=0.97)})

## Use Case: Check Outbound Messages
- Slides 16-17
- Replies support agents and an AI assistant are about to send: email, chat, social
- `gold_contains_personal_data` and `gold_category` are hand labels

In [20]:
outbound = pd.read_csv("data/outbound_messages.csv")
outbound.head()

,id,channel,text,gold_contains_personal_data,gold_category
0,1,email,"Hi Marcus, thanks for reaching out! Your refun...",False,clean
1,2,chat,"No worries at all, glad the reset link worked....",False,clean
2,3,social,Sorry to hear about the delay 😕 Could you DM u...,False,clean
3,4,email,"Hello, our office will be closed on Monday for...",False,clean
4,5,chat,you can update your shipping address any time ...,False,clean


In [21]:
outbound.gold_category.value_counts()

gold_category
clean            29
personal_data    15
internal_info     8
Name: count, dtype: int64

- `criteria` on the Noul says what counts as personal data and what does not

In [22]:
GUARD = {
    "personal": {
        "type": "noul",
        "instructions": "The message contains personal data about a specific person.",
        "criteria": {
            "true": "a phone number, home address, payment card digits, ID or passport "
                    "number, personal email address, date of birth, or health details",
            "false": "no personal data; a first name in a greeting, an order number or "
                     "the company's own helpline does not count",
        },
    },
    "category": {
        "type": "choice",
        "instructions": "What must be checked before this message is sent outside the "
                        "company?",
        "criteria": {
            "clean": "nothing sensitive, safe to send",
            "personal_data": "personal data about a specific person: phone, address, "
                             "card digits, ID numbers, personal email, date of birth, "
                             "health details",
            "internal_info": "internal-only company information: staff discount codes, "
                             "internal links, unreleased plans, internal pricing or "
                             "incidents",
        },
    },
}

r = ollama.systemone(model="nimble", state=outbound.text[5], questions=GUARD)
r.answers

{'personal': SystemOneNoulAnswer(type='noul', noul=0.9933605858194312),
 'category': SystemOneChoiceAnswer(type='choice', choice='personal_data', probabilities={'clean': 0.00760450787128546, 'personal_data': 0.9915491121036076, 'internal_info': 0.0008463800251067681}, confidence=0.9873236681554113)}

## All Messages
- One request per message, both questions in each

In [23]:
start = time.perf_counter()
results = [
    ollama.systemone(model="nimble", state=text, questions=GUARD)
    for text in outbound.text
]
time.perf_counter() - start

3.073357599991141

In [24]:
outbound["p_personal"] = [r.answers["personal"].noul for r in results]
outbound["category"] = [r.answers["category"].choice for r in results]
outbound["confidence"] = [r.answers["category"].confidence for r in results]
outbound[["text", "gold_category", "p_personal", "category", "confidence"]].head(10)

,text,gold_category,p_personal,category,confidence
0,"Hi Marcus, thanks for reaching out! Your refun...",clean,0.028649,clean,0.933791
1,"No worries at all, glad the reset link worked....",clean,0.009419,clean,0.978530
2,Sorry to hear about the delay 😕 Could you DM u...,clean,0.054818,clean,0.933295
3,"Hello, our office will be closed on Monday for...",clean,0.003133,clean,0.925918
4,you can update your shipping address any time ...,clean,0.927514,personal_data,0.902777
5,@ok_ritika I've confirmed your new delivery ad...,personal_data,0.993361,personal_data,0.987324
6,"Hi Ms. Reyes, we tried calling you at 555-0142...",personal_data,0.987403,personal_data,0.976847
7,Good news! Use code STAFF50-INTERNAL at checko...,internal_info,0.001542,internal_info,0.998670
8,"Thanks for the kind words, Jen! We'll pass thi...",clean,0.023482,clean,0.920770
9,"Hi there, the app update (version 4.2) fixes t...",clean,0.003270,clean,0.975500


## Accuracy Against the Hand Labels

In [25]:
((outbound.p_personal > 0.5) == outbound.gold_contains_personal_data).mean()

np.float64(0.9807692307692307)

In [26]:
(outbound.category == outbound.gold_category).mean()

np.float64(0.9807692307692307)

In [27]:
pd.crosstab(outbound.gold_category, outbound.category)

category,clean,internal_info,personal_data
gold_category,,,
clean,28,0,1
internal_info,0,8,0
personal_data,0,0,15


## Reviewing the Wrong Answers

In [28]:
outbound[(outbound.p_personal > 0.5) != outbound.gold_contains_personal_data][
    ["id", "text", "gold_contains_personal_data", "p_personal"]
]

,id,text,gold_contains_personal_data,p_personal
4,5,you can update your shipping address any time ...,False,0.927514


In [29]:
outbound[outbound.category != outbound.gold_category][
    ["id", "text", "gold_category", "category", "confidence"]
]

,id,text,gold_category,category,confidence
4,5,you can update your shipping address any time ...,clean,personal_data,0.902777


## Deciding to Send or Hold a Message
- Slide 18
- Hold when either answer flags the message
- A low threshold on `p_personal` holds more messages and lets fewer leaks through

In [30]:
outbound["hold"] = (outbound.p_personal > 0.3) | (outbound.category != "clean")
outbound["action"] = outbound.hold.map({True: "hold for review", False: "send"})
pd.crosstab(outbound.gold_category, outbound.action)

action,hold for review,send
gold_category,,
clean,1,28
internal_info,8,0
personal_data,15,0


In [31]:
outbound[(outbound.action == "send") & (outbound.gold_category != "clean")][
    ["id", "text", "gold_category", "p_personal", "category"]
]

,id,text,gold_category,p_personal,category


In [32]:
outbound[(outbound.action == "hold for review") & (outbound.gold_category == "clean")][
    ["id", "text", "p_personal", "category"]
]

,id,text,p_personal,category
4,5,you can update your shipping address any time ...,0.927514,personal_data


## Limit: 64 Questions per Request
- Slide 19, covering every limit below
- The server rejects the whole request; `ollama.ResponseError` carries its message

In [33]:
MANY = {"check_" + str(i): GUARD["personal"] for i in range(65)}

try:
    ollama.systemone(model="nimble", state=outbound.text[0], questions=MANY)
except ollama.ResponseError as e:
    error = e
error

ollama._types.ResponseError('questions must contain 1–64 fields')

## Every Question Gets Its Own Prompt
- Slides 20-21
- Each prompt holds the state and every question, so `input_tokens` grows faster than the question count
- `seconds` grows with it

In [34]:
rows = []
for n in [1, 8, 32, 64]:
    start = time.perf_counter()
    r = ollama.systemone(
        model="nimble",
        state=outbound.text[0],
        questions={"check_" + str(i): GUARD["personal"] for i in range(n)},
    )
    rows.append({
        "questions": n,
        "input_tokens": r.usage.input_tokens,
        "seconds": time.perf_counter() - start,
    })
pd.DataFrame(rows)

,questions,input_tokens,seconds
0,1,253,0.042123
1,8,8128,0.120464
2,32,116950,0.815314
3,64,459190,2.966290


## Limit: Prompt Must Fit the Context Window
- Slide 22
- `thread`: every outbound message joined, repeated to make a long email thread
- Both models fit 8,192 tokens per prompt; nothing is cut
- nimble's error says 8,448: its 8,194-token window, rounded up to a multiple of 256

In [35]:
thread = " ".join(outbound.text)
len(thread)

5660

In [36]:
try:
    ollama.systemone(model="nimble", state=thread * 6, questions=GUARD)
except ollama.ResponseError as e:
    error = e
error

ollama._types.ResponseError('request (8995 tokens) exceeds the available context size (8448 tokens), try increasing it')

In [37]:
try:
    ollama.systemone(model="tev1:0.8b", state=thread * 6, questions=GUARD)
except ollama.ResponseError as e:
    error = e
error

ollama._types.ResponseError('request (8778 tokens) exceeds the available context size (8192 tokens), try increasing it')

In [38]:
ollama.systemone(
    model="tev1:0.8b",
    state=thread,
    questions={"personal": GUARD["personal"]},
).usage

SystemOneUsage(input_tokens=1593, output_tokens=0)

## Limit: 64 KB of Text, Text Only
- `status_code` 413: the text and schema are over 64 KB; images do not count toward it
- `images` goes in the raw JSON body; `ollama.systemone` has no images argument

In [39]:
try:
    ollama.systemone(model="nimble", state=thread * 14, questions=GUARD)
except ollama.ResponseError as e:
    error = e
error.status_code, error

(413, ollama._types.ResponseError('text and schema must not exceed 64 KiB'))

In [40]:
PIXEL = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8"
    "z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg=="
)

httpx.post(URL, json={**body, "images": [PIXEL]}).json()

{'error': 'registry.ollama.ai/library/nimble:latest does not support vision'}

## Compare: nimble vs tev1:0.8b vs Hosted Jev
- Slides 23-24
- 200 real card messages, 10 intents; the same `Choice` for every backend
- One message at a time, so `seconds` is per message, end to end

In [41]:
cards = pd.read_csv("data/card_messages.csv")
cards.head()

,id,text,intent
0,1,I'm starting to think my card is lost because ...,card_arrival
1,2,What do I need to do to change my PIN?,change_pin
2,3,What do I do if I think my card was improperly...,compromised_card
3,4,I cannot find my credit card.,lost_or_stolen_card
4,5,How can I change my Tholepin ?,change_pin


In [42]:
INTENT = Choice(
    instructions="Which card problem is the customer writing about?",
    criteria={
        "card_arrival": "a new or replacement card has not arrived yet",
        "activate_my_card": "how to activate a card they already have",
        "change_pin": "changing or resetting the PIN",
        "card_about_to_expire": "the card expires soon, getting a new one",
        "card_not_working": "the card does not work at all",
        "declined_card_payment": "one particular payment was declined",
        "card_payment_fee_charged": "an unexpected fee on a card payment",
        "card_payment_wrong_exchange_rate": "a wrong exchange rate on a card payment in "
                                            "another currency",
        "compromised_card": "someone else may have used the card or its details",
        "lost_or_stolen_card": "the card was lost or stolen",
    },
)

nimble.system_one(cards.text[0], {"intent": INTENT}).answers["intent"]

ChoiceAnswer(type='choice', choice='card_arrival', confidence=0.9809287177620495, probabilities={'card_arrival': 0.9828358459858445, 'activate_my_card': 0.0004772028645970685, 'change_pin': 0.00015691656849709768, 'card_about_to_expire': 0.0004187459744870495, 'card_not_working': 0.00017195776043985806, 'declined_card_payment': 4.826053841255794e-05, 'card_payment_fee_charged': 3.1410410608461436e-05, 'card_payment_wrong_exchange_rate': 1.598006365171018e-05, 'compromised_card': 0.00022123043571668647, 'lost_or_stolen_card': 0.015622449397745167})

In [43]:
def route_all(client):
    start = time.perf_counter()
    answers = [
        client.system_one(text, {"intent": INTENT}).answers["intent"].choice
        for text in cards.text
    ]
    return answers, (time.perf_counter() - start) / len(cards)

In [44]:
cards["nimble"], nimble_seconds = route_all(nimble)
cards["tev1"], tev1_seconds = route_all(tev1)
cards["jev"], jev_seconds = route_all(jev)

In [45]:
pd.DataFrame(
    {
        "accuracy": [
            (cards.nimble == cards.intent).mean(),
            (cards.tev1 == cards.intent).mean(),
            (cards.jev == cards.intent).mean(),
        ],
        "seconds_per_message": [nimble_seconds, tev1_seconds, jev_seconds],
    },
    index=["nimble", "tev1:0.8b", "jev"],
)

,accuracy,seconds_per_message
nimble,0.970,0.052348
tev1:0.8b,0.890,0.027522
jev,0.965,0.758726


## Accuracy per Intent

In [46]:
cards[["nimble", "tev1", "jev"]].eq(cards.intent, axis=0).groupby(cards.intent).mean()

,nimble,tev1,jev
intent,,,
activate_my_card,0.90,0.85,0.95
card_about_to_expire,1.00,0.80,0.85
card_arrival,1.00,0.85,1.00
card_not_working,0.95,0.95,1.00
card_payment_fee_charged,1.00,1.00,1.00
card_payment_wrong_exchange_rate,0.95,0.90,0.95
change_pin,1.00,1.00,1.00
compromised_card,1.00,0.90,1.00
declined_card_payment,1.00,0.65,0.90


## Where nimble and Jev Disagree

In [47]:
cards[(cards.nimble != cards.jev)][["text", "intent", "nimble", "tev1", "jev"]]

,text,intent,nimble,tev1,jev
14,I think I lost my card . I dont know how long ...,lost_or_stolen_card,compromised_card,lost_or_stolen_card,lost_or_stolen_card
29,How can I get my physical card to work?,card_not_working,activate_my_card,card_not_working,card_not_working
41,"Hello, I am facing the issue with my new card ...",declined_card_payment,declined_card_payment,card_not_working,card_not_working
65,Are you able to locate my card?,lost_or_stolen_card,card_arrival,lost_or_stolen_card,lost_or_stolen_card
92,I need to know the cost and when I will receiv...,card_about_to_expire,card_about_to_expire,card_arrival,card_arrival
117,Can I receive a new card while I am in China?,card_about_to_expire,card_about_to_expire,card_arrival,card_arrival
129,"in china, need new card",card_about_to_expire,card_about_to_expire,card_arrival,card_arrival
159,I tried activating my bill of fare and it didn...,activate_my_card,card_not_working,card_not_working,activate_my_card
195,My new card keeps getting declined. I was very...,declined_card_payment,declined_card_payment,card_not_working,card_not_working


## Cheat Sheet

| Need | Code |
|---|---|
| Get a decision model | `ollama pull nimble` |
| Check it is one | `ollama show nimble` (Capabilities: decision) |
| Raw HTTP | `httpx.post("http://localhost:11434/v1/systemone", json={"model", "state", "questions"})` |
| Python client | `ollama.systemone(model="nimble", state=text, questions={...})` |
| Noul | `r.answers["q"].noul` |
| Choice | `r.answers["q"].choice`, `.probabilities`, `.confidence` |
| Score | `r.answers["q"].score`, `.legend`, `.probabilities` |
| Tokens used | `r.usage.input_tokens` |
| TypeSafe SDK on Ollama | `TypeSafeClient(base_url="http://localhost:11434", api_key="local", model="nimble")` |
| Hosted Jev | `load_dotenv(); TypeSafeClient()` |
| Catch a rejected request | `except ollama.ResponseError as e: e.status_code, e.error` |
| Limits | 1 to 64 questions, prompt fit per model (`num_ctx`), 64 KB of text and schema, text only |